# Session-based Binary Prediction with XLNet — Pretrain + Fine-tune

Two-phase training:
1. **Pretrain** — MLM-only loss on session sequences (no labels needed)
2. **Fine-tune** — BCE classification loss + optional MLM regularization

No dependency on transformers4rec or merlin.

In [15]:
import os
import gc
import csv
import glob
import math
import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LambdaLR

from transformers import XLNetConfig, XLNetModel
from tqdm.auto import tqdm

device = torch.device("cuda")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

Using device: cuda
PyTorch version: 2.5.1+cu121


## Configuration

In [ ]:
# ---------------------------------------------------------------------------
# Feature configuration — sequential (per-position in the session)
# ---------------------------------------------------------------------------
CATEGORICAL_FEATURES = {
    "sess_pid_seq":  {"cardinality": 164386, "embedding_dim": 64},
    "sess_csid_seq": {"cardinality": 622,    "embedding_dim": 32},
    "sess_ccid_seq": {"cardinality": 129,    "embedding_dim": 16},
    "sess_bid_seq":  {"cardinality": 3426,   "embedding_dim": 32},
}

CONTINUOUS_FEATURES = [
    "sess_price_log_norm_seq",
    "sess_dtime_secs_log_norm_seq",
    "sess_prod_recency_days_log_norm_seq",
]

# ---------------------------------------------------------------------------
# Feature configuration — static (one value per session, not sequential)
# Leave empty to disable the static branch entirely.
# ---------------------------------------------------------------------------
STATIC_CATEGORICAL_FEATURES = {
    "user_segment": {"cardinality": 10, "embedding_dim": 8},
    "device_type":  {"cardinality": 5,  "embedding_dim": 4},
}

STATIC_CONTINUOUS_FEATURES = [
    "user_lifetime_days",
    "user_avg_order_value",
]

STATIC_PROJ_DIM = 64  # output dim of the static-feature MLP

# ---------------------------------------------------------------------------
# Optional MLM auxiliary loss (set MASK_PROB=0.0 to disable)
# ---------------------------------------------------------------------------
MASK_PROB = 0.15
MLM_WEIGHT = 0.1
MLM_TARGET_FEATURES = [
    "sess_pid_seq",
    "sess_csid_seq",
]

# ---------------------------------------------------------------------------
# Pretraining hyperparameters
# ---------------------------------------------------------------------------
PRETRAIN_MASK_PROB = 0.15         # mask probability during pretraining
PRETRAIN_EPOCHS = 5
PRETRAIN_LEARNING_RATE = 1e-4
PRETRAIN_WEIGHT_DECAY = 1e-5

# Single target: TARGET_COLUMN = "target"
# Multi target:  TARGET_COLUMN = ["target_purchase", "target_cart"]
TARGET_COLUMN = "target"

# Derived — used by dataset, model, and inference cells
TARGET_COLUMNS = TARGET_COLUMN if isinstance(TARGET_COLUMN, list) else [TARGET_COLUMN]
NUM_CLASSES = len(TARGET_COLUMNS)

HAS_STATIC = bool(STATIC_CATEGORICAL_FEATURES) or bool(STATIC_CONTINUOUS_FEATURES)

# ---------------------------------------------------------------------------
# Model hyperparameters
# ---------------------------------------------------------------------------
MAX_SEQ_LEN = 20
D_MODEL = 448
N_HEAD = 8
N_LAYER = 2
CONTINUOUS_PROJ_DIM = 64
CLASSIFIER_HIDDEN_DIM = 128

# ---------------------------------------------------------------------------
# Training hyperparameters
# ---------------------------------------------------------------------------
LEARNING_RATE = 0.00020171456712823088
WEIGHT_DECAY = 2.747484129693843e-05
NUM_EPOCHS = 10
TRAIN_BATCH_SIZE = 256
EVAL_BATCH_SIZE = 512

# ---------------------------------------------------------------------------
# Data paths
# ---------------------------------------------------------------------------
SRC_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "preprocessed_data"))
OUTPUT_DIR = os.path.abspath(os.path.join(os.getcwd(), "data_trimmed"))

START_WINDOW = int(os.environ.get("start_window_index", "1"))
FINAL_WINDOW = int(os.environ.get("final_window_index", "30"))

print(f"Source data:  {SRC_DIR}")
print(f"Trimmed data: {OUTPUT_DIR}")
print(f"Window range: {START_WINDOW} \u2013 {FINAL_WINDOW}")
print(f"Targets ({NUM_CLASSES}): {TARGET_COLUMNS}")
print(f"Static features: {HAS_STATIC}")
print(f"MLM masking: MASK_PROB={MASK_PROB}, targets={MLM_TARGET_FEATURES}")

## Synthetic Data Generation (for testing)

In [17]:
# ---------------------------------------------------------------------------
# Generate synthetic parquet data for testing
# ---------------------------------------------------------------------------
import shutil

rng = np.random.default_rng(42)

N_SESSIONS_TRAIN = 500
N_SESSIONS_EVAL = 100
N_WINDOWS = 3  # generate windows 1..3

# Override window range to match synthetic data
START_WINDOW = 1
FINAL_WINDOW = N_WINDOWS + 1  # train on 1..N_WINDOWS, eval on 2..N_WINDOWS+1

# Clean and recreate output dir
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)


def make_synthetic_df(n_sessions, rng):
    """Create a DataFrame matching the expected schema."""
    rows = []
    for _ in range(n_sessions):
        seq_len = rng.integers(2, MAX_SEQ_LEN + 1)
        row = {"sess_seq_len": int(seq_len)}

        # Sequential categorical features (lists of ints)
        for feat, cfg in CATEGORICAL_FEATURES.items():
            row[feat] = rng.integers(1, cfg["cardinality"], size=seq_len).tolist()

        # Sequential continuous features (lists of floats)
        for feat in CONTINUOUS_FEATURES:
            row[feat] = rng.standard_normal(seq_len).astype(np.float32).tolist()

        # Static categorical features (scalar ints)
        for feat, cfg in STATIC_CATEGORICAL_FEATURES.items():
            row[feat] = int(rng.integers(1, cfg["cardinality"]))

        # Static continuous features (scalar floats)
        for feat in STATIC_CONTINUOUS_FEATURES:
            row[feat] = float(rng.standard_normal())

        # Target (binary, ~20% positive rate)
        row["target"] = int(rng.random() < 0.2)

        rows.append(row)
    return pd.DataFrame(rows)


for w in range(1, N_WINDOWS + 2):  # +2 so last window has eval/test data
    window_dir = os.path.join(OUTPUT_DIR, f"{w:04d}")
    os.makedirs(window_dir, exist_ok=True)

    train_df = make_synthetic_df(N_SESSIONS_TRAIN, rng)
    valid_df = make_synthetic_df(N_SESSIONS_EVAL, rng)
    test_df_synth = make_synthetic_df(N_SESSIONS_EVAL, rng)

    train_df.to_parquet(os.path.join(window_dir, "train.parquet"), index=False)
    valid_df.to_parquet(os.path.join(window_dir, "valid.parquet"), index=False)
    test_df_synth.to_parquet(os.path.join(window_dir, "test.parquet"), index=False)

print(f"Generated {N_WINDOWS + 1} windows in {OUTPUT_DIR}")
print(f"  Train: {N_SESSIONS_TRAIN} sessions, Eval/Test: {N_SESSIONS_EVAL} sessions each")
print(f"  Window range overridden: START_WINDOW={START_WINDOW}, FINAL_WINDOW={FINAL_WINDOW}")

# Also generate test.pkl for the inference cell
test_pkl_df = make_synthetic_df(200, rng)
test_pkl_df.to_pickle(os.path.join(os.getcwd(), "test.pkl"))
print(f"Generated test.pkl with {len(test_pkl_df)} sessions")

Generated 4 windows in c:\Users\gongz\transformers4rec_experiment\t4r_torch\data_trimmed
  Train: 500 sessions, Eval/Test: 100 sessions each
  Window range overridden: START_WINDOW=1, FINAL_WINDOW=4
Generated test.pkl with 200 sessions


In [18]:
_preview = pd.read_parquet(os.path.join(OUTPUT_DIR, "0001/train.parquet"))
print(f"Shape: {_preview.shape}")
print(f"Columns: {list(_preview.columns)}\n")
_preview.head(5)

Shape: (500, 13)
Columns: ['sess_seq_len', 'sess_pid_seq', 'sess_csid_seq', 'sess_ccid_seq', 'sess_bid_seq', 'sess_price_log_norm_seq', 'sess_dtime_secs_log_norm_seq', 'sess_prod_recency_days_log_norm_seq', 'user_segment', 'device_type', 'user_lifetime_days', 'user_avg_order_value', 'target']



,sess_seq_len,sess_pid_seq,sess_csid_seq,sess_ccid_seq,sess_bid_seq,sess_price_log_norm_seq,sess_dtime_secs_log_norm_seq,sess_prod_recency_days_log_norm_seq,user_segment,device_type,user_lifetime_days,user_avg_order_value,target
0,3,"[127227, 107602, 72146]","[269, 534, 54]","[90, 26, 13]","[1804, 3342, 2520]","[-0.31624260544776917, -0.01680115796625614, -...","[0.879397988319397, 0.7777919173240662, 0.0660...","[1.1272412538528442, 0.46750932931900024, -0.8...",7,1,-0.958883,0.878450,0
1,12,"[27162, 124619, 115156, 58279, 11166, 159569, ...","[226, 290, 310, 28, 340, 96, 462, 425, 573, 46...","[53, 42, 116, 48, 10, 61, 102, 25, 60, 17, 88,...","[1131, 778, 1934, 2295, 3221, 1498, 551, 2852,...","[-0.6655097007751465, 0.23216132819652557, 0.1...","[-0.47037264704704285, -0.6388778686523438, -0...","[-0.3487250804901123, -0.46235179901123047, 0....",9,3,-0.309347,0.456775,1
2,8,"[3734, 17449, 14803, 126832, 118746, 114539, 7...","[101, 560, 312, 583, 95, 309, 433, 308]","[58, 22, 49, 31, 39, 88, 81, 78]","[1240, 3287, 301, 1174, 405, 1161, 3295, 1255]","[-0.9054790735244751, -0.37816256284713745, 1....","[-0.3390330672264099, 0.8403081297874451, -1.7...","[-0.5294927358627319, 0.2326762080192566, 0.02...",9,1,0.835111,0.356871,1
3,11,"[28798, 96600, 130437, 28043, 49315, 152076, 1...","[367, 184, 15, 95, 596, 260, 300, 291, 487, 41...","[36, 63, 86, 63, 57, 121, 21, 74, 70, 61, 104]","[915, 3267, 1136, 1867, 1784, 2911, 1504, 975,...","[0.1570485681295395, -0.15863484144210815, -1....","[-0.9651167392730713, -0.7252260446548462, 2.1...","[0.44607219099998474, -0.4549834728240967, -1....",4,3,1.106289,0.428756,0
4,9,"[43145, 873, 153999, 155793, 39612, 5246, 2018...","[554, 96, 464, 112, 229, 373, 448, 544, 28]","[26, 36, 40, 128, 100, 42, 125, 64, 65]","[1486, 493, 3201, 48, 597, 787, 957, 452, 62]","[-2.0501720905303955, -0.0487184002995491, -0....","[-0.4841694235801697, -0.3276731073856354, 1.0...","[0.17657335102558136, -1.084388017654419, 0.09...",7,3,0.315605,1.205854,0


## Dataset

In [ ]:
class SessionDataset(Dataset):
    """Loads parquet files and converts variable-length sessions into
    fixed-length tensors (padded / truncated to MAX_SEQ_LEN)."""

    def __init__(self, parquet_paths, max_seq_len=MAX_SEQ_LEN):
        dfs = [pd.read_parquet(p) for p in parquet_paths]
        self.df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq_len = min(int(row["sess_seq_len"]), self.max_seq_len)
        result = {}

        # --- Sequential categorical features ---
        for feat_name in CATEGORICAL_FEATURES:
            arr = np.array(row[feat_name], dtype=np.int64)[: self.max_seq_len]
            padded = np.zeros(self.max_seq_len, dtype=np.int64)
            padded[: len(arr)] = arr
            result[feat_name] = torch.tensor(padded, dtype=torch.long)

        # --- Sequential continuous features ---
        cont_arrays = []
        for feat_name in CONTINUOUS_FEATURES:
            arr = np.array(row[feat_name], dtype=np.float32)[: self.max_seq_len]
            padded = np.zeros(self.max_seq_len, dtype=np.float32)
            padded[: len(arr)] = arr
            cont_arrays.append(padded)
        result["continuous"] = torch.tensor(
            np.stack(cont_arrays, axis=-1), dtype=torch.float32
        )

        # --- Attention mask ---
        mask = np.zeros(self.max_seq_len, dtype=np.float32)
        mask[:seq_len] = 1.0
        result["attention_mask"] = torch.tensor(mask, dtype=torch.float32)

        # --- Static categorical features (scalars, not sequences) ---
        for feat_name in STATIC_CATEGORICAL_FEATURES:
            result[f"static_{feat_name}"] = torch.tensor(
                int(row[feat_name]), dtype=torch.long
            )

        # --- Static continuous features ---
        if STATIC_CONTINUOUS_FEATURES:
            static_cont = np.array(
                [float(row[f]) for f in STATIC_CONTINUOUS_FEATURES], dtype=np.float32
            )
            result["static_continuous"] = torch.tensor(static_cont, dtype=torch.float32)

        # --- Target ---
        if NUM_CLASSES > 1:
            result["target"] = torch.tensor(
                [float(row[c]) for c in TARGET_COLUMNS], dtype=torch.float32
            )  # (NUM_CLASSES,)
        else:
            result["target"] = torch.tensor(
                float(row[TARGET_COLUMNS[0]]), dtype=torch.float32
            )  # scalar

        return result


_sample_paths = glob.glob(os.path.join(OUTPUT_DIR, "0001/train.parquet"))
_ds = SessionDataset(_sample_paths)
_item = _ds[0]
print(f"Dataset size: {len(_ds)}")
for k, v in _item.items():
    print(f"  {k}: {v.shape} {v.dtype}")
del _ds, _item

## Model Architecture

In [ ]:
class SessionXLNetModel(nn.Module):
    """XLNet-based session model with a binary/multi-label classification head.
    Optionally incorporates static features and MLM auxiliary loss.
    Supports pretrain mode (MLM-only, no classifier)."""

    def __init__(self):
        super().__init__()

        # === Sequential feature embeddings ===
        self.embeddings = nn.ModuleDict()
        for feat, cfg in CATEGORICAL_FEATURES.items():
            self.embeddings[feat] = nn.Embedding(
                cfg["cardinality"], cfg["embedding_dim"], padding_idx=0
            )

        self.continuous_proj = nn.Sequential(
            nn.Linear(len(CONTINUOUS_FEATURES), CONTINUOUS_PROJ_DIM),
            nn.ReLU(),
        )

        cat_dim = sum(c["embedding_dim"] for c in CATEGORICAL_FEATURES.values())
        total_input_dim = cat_dim + CONTINUOUS_PROJ_DIM
        self.input_mlp = nn.Sequential(
            nn.Linear(total_input_dim, D_MODEL),
            nn.ReLU(),
        )

        # === XLNet backbone ===
        xlnet_config = XLNetConfig(
            vocab_size=2,
            d_model=D_MODEL,
            n_layer=N_LAYER,
            n_head=N_HEAD,
            d_inner=4 * D_MODEL,
            ff_activation="gelu",
            attn_type="bi",
            mem_len=0,
            dropout=0.0,
        )
        self.xlnet = XLNetModel(xlnet_config)

        # === Static feature branch (optional) ===
        classifier_input_dim = D_MODEL

        if HAS_STATIC:
            self.static_embeddings = nn.ModuleDict()
            for feat, cfg in STATIC_CATEGORICAL_FEATURES.items():
                self.static_embeddings[feat] = nn.Embedding(
                    cfg["cardinality"], cfg["embedding_dim"], padding_idx=0
                )

            static_cat_dim = sum(
                c["embedding_dim"] for c in STATIC_CATEGORICAL_FEATURES.values()
            )
            static_cont_dim = len(STATIC_CONTINUOUS_FEATURES)
            static_raw_dim = static_cat_dim + static_cont_dim

            self.static_mlp = nn.Sequential(
                nn.Linear(static_raw_dim, STATIC_PROJ_DIM),
                nn.ReLU(),
            )

            # LayerNorm on each branch before concat
            self.seq_norm = nn.LayerNorm(D_MODEL)
            self.static_norm = nn.LayerNorm(STATIC_PROJ_DIM)

            classifier_input_dim = D_MODEL + STATIC_PROJ_DIM

        # === MLM auxiliary loss (optional) ===
        self.mlm_heads = nn.ModuleDict()
        if MLM_TARGET_FEATURES:
            self.mask_embedding = nn.Parameter(torch.randn(D_MODEL))
            for feat in MLM_TARGET_FEATURES:
                card = CATEGORICAL_FEATURES[feat]["cardinality"]
                self.mlm_heads[feat] = nn.Linear(D_MODEL, card)
            self.mlm_loss_fn = nn.CrossEntropyLoss()

        # === Classification head ===
        self.classifier = nn.Sequential(
            nn.Linear(classifier_input_dim, CLASSIFIER_HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(CLASSIFIER_HIDDEN_DIM, NUM_CLASSES),
        )

        self.loss_fn = nn.BCEWithLogitsLoss()

    def forward(self, batch, pretrain=False):
        # --- Sequential embeddings ---
        cat_embeds = [self.embeddings[f](batch[f]) for f in CATEGORICAL_FEATURES]
        cont_proj = self.continuous_proj(batch["continuous"])

        x = torch.cat(cat_embeds + [cont_proj], dim=-1)
        x = self.input_mlp(x)  # (B, T, D_MODEL)

        attention_mask = batch["attention_mask"]

        # --- MLM masking (always in pretrain; optional in fine-tune) ---
        mlm_loss = None
        effective_mask_prob = PRETRAIN_MASK_PROB if pretrain else MASK_PROB
        if self.training and effective_mask_prob > 0 and self.mlm_heads:
            mask_probs = effective_mask_prob * attention_mask  # (B, T)
            mlm_mask = torch.bernoulli(mask_probs).bool()  # (B, T)

            if mlm_mask.any():
                mlm_targets = {}
                for feat in MLM_TARGET_FEATURES:
                    mlm_targets[feat] = batch[feat][mlm_mask]

                x = x.clone()
                x[mlm_mask] = self.mask_embedding

        # --- XLNet forward ---
        xlnet_out = self.xlnet(inputs_embeds=x, attention_mask=attention_mask)
        hidden = xlnet_out.last_hidden_state  # (B, T, D_MODEL)

        # --- MLM loss computation ---
        if self.training and effective_mask_prob > 0 and self.mlm_heads:
            if mlm_mask.any():
                masked_hidden = hidden[mlm_mask]
                feat_losses = []
                for feat in MLM_TARGET_FEATURES:
                    mlm_logits = self.mlm_heads[feat](masked_hidden)
                    feat_losses.append(
                        self.mlm_loss_fn(mlm_logits, mlm_targets[feat])
                    )
                mlm_loss = torch.stack(feat_losses).mean()

        # --- Pretrain mode: return MLM loss only, skip classifier ---
        if pretrain:
            if mlm_loss is None:
                raise ValueError("pretrain=True but no MLM loss computed. "
                                 "Check PRETRAIN_MASK_PROB and MLM_TARGET_FEATURES.")
            return mlm_loss, None

        # --- Masked mean pooling ---
        mask_expanded = attention_mask.unsqueeze(-1)  # (B, T, 1)
        hidden_masked = hidden * mask_expanded
        pooled = hidden_masked.sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1)

        # --- Combine with static features ---
        if HAS_STATIC:
            pooled = self.seq_norm(pooled)

            static_parts = []
            for feat in STATIC_CATEGORICAL_FEATURES:
                static_parts.append(
                    self.static_embeddings[feat](batch[f"static_{feat}"])
                )
            if STATIC_CONTINUOUS_FEATURES:
                static_parts.append(batch["static_continuous"])

            static_cat = torch.cat(static_parts, dim=-1)
            static_repr = self.static_mlp(static_cat)
            static_repr = self.static_norm(static_repr)

            pooled = torch.cat([pooled, static_repr], dim=-1)

        # --- Classifier ---
        logits = self.classifier(pooled)
        if NUM_CLASSES == 1:
            logits = logits.squeeze(-1)
        targets = batch["target"]
        loss = self.loss_fn(logits, targets)

        # Add MLM auxiliary loss during fine-tuning
        if mlm_loss is not None:
            loss = loss + MLM_WEIGHT * mlm_loss

        return loss, logits


model = SessionXLNetModel().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")
if HAS_STATIC:
    n_static = sum(
        p.numel() for n, p in model.named_parameters()
        if "static" in n
    )
    print(f"  Static branch: {n_static:,}")
if model.mlm_heads:
    n_mlm = sum(
        p.numel() for n, p in model.named_parameters()
        if "mlm" in n or "mask_embedding" in n
    )
    print(f"  MLM heads: {n_mlm:,}")

## Evaluation Metrics (Binary Classification)

In [ ]:
from sklearn.metrics import roc_auc_score


def compute_auc(logits, targets):
    """Compute AUC from raw logits and targets.
    Returns a dict of per-target AUCs when NUM_CLASSES > 1, or a single float.
    """
    probs = torch.sigmoid(logits).cpu().numpy()
    targets_np = targets.cpu().numpy()

    if NUM_CLASSES > 1:
        auc_dict = {}
        for i, col in enumerate(TARGET_COLUMNS):
            try:
                auc_dict[col] = roc_auc_score(targets_np[:, i], probs[:, i])
            except ValueError:
                auc_dict[col] = float("nan")
        return auc_dict
    else:
        try:
            return roc_auc_score(targets_np, probs)
        except ValueError:
            return float("nan")

## Training & Evaluation Helpers

In [ ]:
# ---------------------------------------------------------------------------
# CSV logger
# ---------------------------------------------------------------------------
LOG_FIELDS = [
    "timestamp", "phase", "day", "epoch", "batch", "loss", "auc",
]


def init_log(path):
    with open(path, "w", newline="") as f:
        csv.writer(f).writerow(LOG_FIELDS)
    return path


def log_row(path, **kwargs):
    with open(path, "a", newline="") as f:
        csv.writer(f).writerow([kwargs.get(c, "") for c in LOG_FIELDS])


# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------
def train_one_day(model, train_loader, optimizer, scheduler, log_path, day):
    model.train()
    epoch_bar = tqdm(range(NUM_EPOCHS), desc="Epochs", unit="epoch")
    for epoch in epoch_bar:
        epoch_loss = 0.0
        n_batches = 0
        batch_bar = tqdm(train_loader, desc=f"  Epoch {epoch+1}/{NUM_EPOCHS}",
                         leave=False, unit="batch")
        for batch in batch_bar:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            loss, _ = model(batch)
            loss.backward()
            optimizer.step()
            scheduler.step()

            batch_loss = loss.item()
            epoch_loss += batch_loss
            n_batches += 1
            batch_bar.set_postfix(batch_loss=f"{batch_loss:.4f}")

            log_row(log_path, timestamp=datetime.datetime.now().isoformat(),
                    phase="train", day=day, epoch=epoch + 1,
                    batch=n_batches, loss=f"{batch_loss:.6f}")

        avg_epoch_loss = epoch_loss / max(n_batches, 1)
        epoch_bar.set_postfix(epoch_loss=f"{avg_epoch_loss:.4f}")
        print(f"    Epoch {epoch+1}/{NUM_EPOCHS}  loss={avg_epoch_loss:.4f}")


# ---------------------------------------------------------------------------
# Evaluation
# ---------------------------------------------------------------------------
def evaluate(model, eval_loader, log_path=None, day=None):
    model.eval()
    total_loss = 0.0
    n_batches = 0
    all_logits = []
    all_targets = []

    with torch.no_grad():
        for batch in tqdm(eval_loader, desc="  Evaluating", leave=False, unit="batch"):
            batch = {k: v.to(device) for k, v in batch.items()}
            loss, logits = model(batch)

            total_loss += loss.item()
            n_batches += 1
            all_logits.append(logits.cpu())
            all_targets.append(batch["target"].cpu())

    all_logits = torch.cat(all_logits)
    all_targets = torch.cat(all_targets)
    auc = compute_auc(all_logits, all_targets)
    avg_loss = total_loss / max(n_batches, 1)

    metrics = {"loss": avg_loss}
    if isinstance(auc, dict):
        for col, val in auc.items():
            metrics[f"auc_{col}"] = val
        metrics["auc_mean"] = np.nanmean(list(auc.values()))
    else:
        metrics["auc"] = auc

    if log_path is not None:
        auc_str = (f"{metrics.get('auc_mean', metrics.get('auc', float('nan'))):.6f}")
        log_row(log_path,
                timestamp=datetime.datetime.now().isoformat(),
                phase="eval", day=day,
                loss=f"{avg_loss:.6f}", auc=auc_str)

    return metrics


def wipe_memory():
    gc.collect()

## Phase 1: Pretraining (MLM only)

Train the transformer backbone with MLM loss only (no labels).  
This teaches the model to understand session sequence context before fine-tuning.

In [ ]:
assert MLM_TARGET_FEATURES, "MLM_TARGET_FEATURES must be non-empty for pretraining"
assert PRETRAIN_MASK_PROB > 0, "PRETRAIN_MASK_PROB must be > 0 for pretraining"

time_stamp_pt = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M")
pretrain_log_path = init_log(f"pretrain_log_{time_stamp_pt}.csv")
print(f"Pretrain log: {pretrain_log_path}")
print(f"Mask prob: {PRETRAIN_MASK_PROB}, Targets: {MLM_TARGET_FEATURES}")
print(f"Epochs: {PRETRAIN_EPOCHS}")

# Collect all training data across windows for pretraining
pretrain_paths = []
for ti in range(START_WINDOW, FINAL_WINDOW + 1):
    pretrain_paths.extend(
        glob.glob(os.path.join(OUTPUT_DIR, f"{ti:04d}/train.parquet"))
    )
if not pretrain_paths:
    raise FileNotFoundError("No training parquets found for pretraining")

pretrain_ds = SessionDataset(pretrain_paths)
pretrain_loader = DataLoader(
    pretrain_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True, drop_last=False,
)
print(f"Pretrain data: {len(pretrain_ds)} sessions from {len(pretrain_paths)} files")

pretrain_optimizer = torch.optim.AdamW(
    model.parameters(), lr=PRETRAIN_LEARNING_RATE, weight_decay=PRETRAIN_WEIGHT_DECAY
)
total_pretrain_steps = len(pretrain_loader) * PRETRAIN_EPOCHS
pretrain_scheduler = LambdaLR(
    pretrain_optimizer,
    lr_lambda=lambda step: max(0.0, 1 - step / max(total_pretrain_steps, 1)),
)

model.train()
for epoch in tqdm(range(PRETRAIN_EPOCHS), desc="Pretrain epochs", unit="epoch"):
    epoch_loss = 0.0
    n_batches = 0
    batch_bar = tqdm(pretrain_loader, desc=f"  Epoch {epoch+1}/{PRETRAIN_EPOCHS}",
                     leave=False, unit="batch")
    for batch in batch_bar:
        batch = {k: v.to(device) for k, v in batch.items()}
        pretrain_optimizer.zero_grad()
        mlm_loss, _ = model(batch, pretrain=True)
        mlm_loss.backward()
        pretrain_optimizer.step()
        pretrain_scheduler.step()

        batch_loss = mlm_loss.item()
        epoch_loss += batch_loss
        n_batches += 1
        batch_bar.set_postfix(mlm_loss=f"{batch_loss:.4f}")

        log_row(pretrain_log_path,
                timestamp=datetime.datetime.now().isoformat(),
                phase="pretrain", epoch=epoch + 1,
                batch=n_batches, loss=f"{batch_loss:.6f}")

    avg_loss = epoch_loss / max(n_batches, 1)
    print(f"    Pretrain epoch {epoch+1}/{PRETRAIN_EPOCHS}  mlm_loss={avg_loss:.4f}")

del pretrain_ds, pretrain_loader
wipe_memory()
print("\nPretraining complete. Proceeding to fine-tuning...")

### Pretrain Evaluation

Evaluate the pretrained transformer by masking random positions in held-out data and measuring MLM loss.

In [ ]:
# Evaluate pretrain quality on held-out test splits
model.eval()
pretrain_eval_paths = []
for ti in range(START_WINDOW, FINAL_WINDOW + 1):
    pretrain_eval_paths.extend(
        glob.glob(os.path.join(OUTPUT_DIR, f"{ti:04d}/test.parquet"))
    )
assert pretrain_eval_paths, "No test parquets found for pretrain evaluation"

pretrain_eval_ds = SessionDataset(pretrain_eval_paths)
pretrain_eval_loader = DataLoader(
    pretrain_eval_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, drop_last=False,
)
print(f"Pretrain eval: {len(pretrain_eval_ds)} sessions from {len(pretrain_eval_paths)} files")

total_mlm_loss = 0.0
total_correct = 0
total_masked = 0
n_batches = 0

# Per-feature tracking
feat_correct = {f: 0 for f in MLM_TARGET_FEATURES}
feat_total = {f: 0 for f in MLM_TARGET_FEATURES}

with torch.no_grad():
    for batch in tqdm(pretrain_eval_loader, desc="  Pretrain eval", leave=False, unit="batch"):
        batch = {k: v.to(device) for k, v in batch.items()}
        attention_mask = batch["attention_mask"]

        # --- Embed & mask (same logic as forward, but deterministic eval) ---
        cat_embeds = [model.embeddings[f](batch[f]) for f in CATEGORICAL_FEATURES]
        cont_proj = model.continuous_proj(batch["continuous"])
        x = torch.cat(cat_embeds + [cont_proj], dim=-1)
        x = model.input_mlp(x)

        # Random masking on valid positions
        mask_probs = PRETRAIN_MASK_PROB * attention_mask
        mlm_mask = torch.bernoulli(mask_probs).bool()

        if not mlm_mask.any():
            continue

        mlm_targets = {}
        for feat in MLM_TARGET_FEATURES:
            mlm_targets[feat] = batch[feat][mlm_mask]

        x_masked = x.clone()
        x_masked[mlm_mask] = model.mask_embedding

        # --- Forward through XLNet ---
        xlnet_out = model.xlnet(inputs_embeds=x_masked, attention_mask=attention_mask)
        hidden = xlnet_out.last_hidden_state
        masked_hidden = hidden[mlm_mask]

        # --- Compute MLM loss and accuracy per feature ---
        batch_losses = []
        for feat in MLM_TARGET_FEATURES:
            mlm_logits = model.mlm_heads[feat](masked_hidden)
            targets = mlm_targets[feat]
            loss = model.mlm_loss_fn(mlm_logits, targets)
            batch_losses.append(loss)

            preds = mlm_logits.argmax(dim=-1)
            n_correct = (preds == targets).sum().item()
            feat_correct[feat] += n_correct
            feat_total[feat] += targets.numel()

        total_mlm_loss += torch.stack(batch_losses).mean().item()
        n_batches += 1

avg_mlm_loss = total_mlm_loss / max(n_batches, 1)
print(f"
Pretrain eval MLM loss: {avg_mlm_loss:.4f}")
print(f"Per-feature accuracy (masked position prediction):")
for feat in MLM_TARGET_FEATURES:
    acc = feat_correct[feat] / max(feat_total[feat], 1)
    card = CATEGORICAL_FEATURES[feat]["cardinality"]
    random_baseline = 1.0 / card
    print(f"  {feat}: {acc:.4%}  (random baseline: {random_baseline:.4%}, vocab={card:,})")

del pretrain_eval_ds, pretrain_eval_loader
wipe_memory()

## Phase 2: Fine-tuning (BCE + optional MLM)

Fine-tune the pretrained model on the classification task.  
The sliding-window and single-window cells below use the same model instance that was pretrained above.

## Sliding-Window Training

In [ ]:
time_stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M")
log_path = init_log(f"training_log_cpu_{time_stamp}.csv")
print(f"Logging to: {log_path}")

for time_index in range(START_WINDOW, FINAL_WINDOW):
    train_paths = glob.glob(
        os.path.join(OUTPUT_DIR, f"{time_index:04d}/train.parquet")
    )
    eval_day = time_index + 1
    eval_paths = glob.glob(
        os.path.join(OUTPUT_DIR, f"{eval_day:04d}/valid.parquet")
    )
    if not train_paths:
        print(f"Day {time_index}: no train data, skipping.")
        continue

    print("=" * 50)
    print(f"Day {time_index}: training  (eval on day {eval_day})")
    print("=" * 50)

    train_ds = SessionDataset(train_paths)
    train_loader = DataLoader(
        train_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True, drop_last=False,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    total_steps = len(train_loader) * NUM_EPOCHS
    scheduler = LambdaLR(
        optimizer, lr_lambda=lambda step: max(0.0, 1 - step / max(total_steps, 1))
    )

    train_one_day(model, train_loader, optimizer, scheduler,
                  log_path, day=time_index)

    if eval_paths:
        eval_ds = SessionDataset(eval_paths)
        eval_loader = DataLoader(
            eval_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False,
        )
        eval_metrics = evaluate(model, eval_loader,
                                log_path=log_path, day=eval_day)
        print(f"  Eval results (day {eval_day}):")
        for k in sorted(eval_metrics):
            print(f"    {k} = {eval_metrics[k]:.6f}")
    else:
        print(f"  No eval data for day {eval_day}.")

    wipe_memory()

print("\nTraining complete.")

## Final Evaluation on Test Splits

In [ ]:
results = []

for day in range(START_WINDOW, FINAL_WINDOW + 1):
    test_paths = glob.glob(
        os.path.join(OUTPUT_DIR, f"{day:04d}/test.parquet")
    )
    if not test_paths:
        continue
    test_ds = SessionDataset(test_paths)
    test_loader = DataLoader(
        test_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False,
    )
    metrics = evaluate(model, test_loader, log_path=log_path, day=day)
    metrics["time_index"] = day
    results.append(metrics)
    print(f"Day {day}:", {k: f"{v:.4f}" for k, v in metrics.items() if k != "time_index"})

results_df = pd.DataFrame(results).set_index("time_index")
results_df.loc["overall"] = results_df.mean()

print("\n" + "=" * 60)
print("Results Summary:")
print(results_df.to_string())

results_df.to_csv(f"evaluation_results_cpu_{time_stamp}.csv")
print(f"\nSaved to evaluation_results_cpu_{time_stamp}.csv")
print(f"Full training log: {log_path}")

## Single-Window Train & Evaluate

Train on one window's training split, evaluate on one window's test split.
Useful for quick iteration, hyperparameter search, or debugging.

In [ ]:
# ---------------------------------------------------------------------------
# Config — pick which window to use
# ---------------------------------------------------------------------------
SINGLE_TRAIN_WINDOW = 1   # window index for training
SINGLE_EVAL_WINDOW = 2    # window index for evaluation

# ---------------------------------------------------------------------------
# Build data loaders
# ---------------------------------------------------------------------------
_train_paths = glob.glob(os.path.join(OUTPUT_DIR, f"{SINGLE_TRAIN_WINDOW:04d}/train.parquet"))
_eval_paths = glob.glob(os.path.join(OUTPUT_DIR, f"{SINGLE_EVAL_WINDOW:04d}/test.parquet"))

assert _train_paths, f"No train data for window {SINGLE_TRAIN_WINDOW}"
assert _eval_paths,  f"No test data for window {SINGLE_EVAL_WINDOW}"

single_train_ds = SessionDataset(_train_paths)
single_eval_ds = SessionDataset(_eval_paths)

single_train_loader = DataLoader(
    single_train_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True, drop_last=False,
)
single_eval_loader = DataLoader(
    single_eval_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False,
)

print(f"Train: window {SINGLE_TRAIN_WINDOW} — {len(single_train_ds)} sessions")
print(f"Eval:  window {SINGLE_EVAL_WINDOW} — {len(single_eval_ds)} sessions")

# ---------------------------------------------------------------------------
# Fresh model + optimizer
# ---------------------------------------------------------------------------
single_model = SessionXLNetModel().to(device)

single_optimizer = torch.optim.AdamW(
    single_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
total_steps = len(single_train_loader) * NUM_EPOCHS
single_scheduler = LambdaLR(
    single_optimizer,
    lr_lambda=lambda step: max(0.0, 1 - step / max(total_steps, 1)),
)

# ---------------------------------------------------------------------------
# Train
# ---------------------------------------------------------------------------
time_stamp_single = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M")
single_log_path = init_log(f"training_log_single_{time_stamp_single}.csv")

print(f"\nTraining for {NUM_EPOCHS} epochs...")
train_one_day(
    single_model, single_train_loader, single_optimizer, single_scheduler,
    single_log_path, day=SINGLE_TRAIN_WINDOW,
)

# ---------------------------------------------------------------------------
# Evaluate
# ---------------------------------------------------------------------------
single_metrics = evaluate(
    single_model, single_eval_loader,
    log_path=single_log_path, day=SINGLE_EVAL_WINDOW,
)

print(f"\nEval results (window {SINGLE_EVAL_WINDOW}):")
for k in sorted(single_metrics):
    print(f"  {k} = {single_metrics[k]:.6f}")

## Visualization

In [ ]:
import matplotlib.pyplot as plt

df_plot = results_df.drop("overall", errors="ignore").copy()
df_plot.index = pd.to_numeric(df_plot.index)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(df_plot.index, df_plot["loss"], marker="o", markersize=5)
axes[0].set_xlabel("Day")
axes[0].set_ylabel("Loss")
axes[0].set_title("Test Loss Over Time")

axes[1].plot(df_plot.index, df_plot["auc"], marker="s", markersize=5, color="tab:orange")
axes[1].set_xlabel("Day")
axes[1].set_ylabel("AUC")
axes[1].set_title("Test AUC Over Time")

plt.suptitle("Binary Classification — CPU", fontsize=14)
plt.tight_layout()
plt.savefig(f"evaluation_metrics_over_time_cpu_{time_stamp}.png", bbox_inches="tight")
plt.show()

## Save Model

In [ ]:
model_path = os.path.join(os.getcwd(), "saved_model")
os.makedirs(model_path, exist_ok=True)
torch.save(model.state_dict(), os.path.join(model_path, "model.pt"))
print(f"Model saved to {model_path}/model.pt")

## Production-Style Inference from test.pkl

In [ ]:
import pickle

# ---------------------------------------------------------------------------
# 1. Load test.pkl
# ---------------------------------------------------------------------------
TEST_PKL_PATH = os.path.join(os.getcwd(), "test.pkl")
with open(TEST_PKL_PATH, "rb") as f:
    test_df = pickle.load(f)
print(f"Loaded {len(test_df)} sessions from {TEST_PKL_PATH}")
print(f"Target columns ({NUM_CLASSES}): {TARGET_COLUMNS}")

# ---------------------------------------------------------------------------
# 2. Build tensors for inference (same logic as SessionDataset.__getitem__)
# ---------------------------------------------------------------------------
def df_to_batch(df, max_seq_len=MAX_SEQ_LEN):
    """Convert a DataFrame of sessions into a batched tensor dict."""
    all_cat = {f: [] for f in CATEGORICAL_FEATURES}
    all_cont = []
    all_mask = []

    # Static feature accumulators
    all_static_cat = {f: [] for f in STATIC_CATEGORICAL_FEATURES}
    all_static_cont = []

    for _, row in df.iterrows():
        seq_len = min(int(row["sess_seq_len"]), max_seq_len)

        for feat_name in CATEGORICAL_FEATURES:
            arr = np.array(row[feat_name], dtype=np.int64)[:max_seq_len]
            padded = np.zeros(max_seq_len, dtype=np.int64)
            padded[:len(arr)] = arr
            all_cat[feat_name].append(padded)

        cont_arrays = []
        for feat_name in CONTINUOUS_FEATURES:
            arr = np.array(row[feat_name], dtype=np.float32)[:max_seq_len]
            padded = np.zeros(max_seq_len, dtype=np.float32)
            padded[:len(arr)] = arr
            cont_arrays.append(padded)
        all_cont.append(np.stack(cont_arrays, axis=-1))

        mask = np.zeros(max_seq_len, dtype=np.float32)
        mask[:seq_len] = 1.0
        all_mask.append(mask)

        # Static features
        for feat_name in STATIC_CATEGORICAL_FEATURES:
            all_static_cat[feat_name].append(int(row[feat_name]))
        if STATIC_CONTINUOUS_FEATURES:
            all_static_cont.append(
                [float(row[f]) for f in STATIC_CONTINUOUS_FEATURES]
            )

    batch = {}
    for feat_name in CATEGORICAL_FEATURES:
        batch[feat_name] = torch.tensor(np.stack(all_cat[feat_name]), dtype=torch.long)
    batch["continuous"] = torch.tensor(np.stack(all_cont), dtype=torch.float32)
    batch["attention_mask"] = torch.tensor(np.stack(all_mask), dtype=torch.float32)

    # Static features
    for feat_name in STATIC_CATEGORICAL_FEATURES:
        batch[f"static_{feat_name}"] = torch.tensor(
            all_static_cat[feat_name], dtype=torch.long
        )
    if STATIC_CONTINUOUS_FEATURES:
        batch["static_continuous"] = torch.tensor(
            all_static_cont, dtype=torch.float32
        )

    # Dummy target — only needed so forward() doesn't error
    batch["target"] = torch.zeros(len(df), NUM_CLASSES, dtype=torch.float32) if NUM_CLASSES > 1 \
        else torch.zeros(len(df), dtype=torch.float32)
    return batch

# ---------------------------------------------------------------------------
# 3. Run inference in mini-batches
# ---------------------------------------------------------------------------
model.eval()
all_logits_list = []

for start in tqdm(range(0, len(test_df), EVAL_BATCH_SIZE), desc="Inference"):
    end = min(start + EVAL_BATCH_SIZE, len(test_df))
    chunk_df = test_df.iloc[start:end]
    batch = df_to_batch(chunk_df)
    batch = {k: v.to(device) for k, v in batch.items()}

    with torch.no_grad():
        _, logits = model(batch)

    all_logits_list.append(logits.cpu().numpy())

# all_logits_out shape: (N,) for single target, (N, NUM_CLASSES) for multi
all_logits_out = np.concatenate(all_logits_list).astype(np.float32)
all_probs = 1.0 / (1.0 + np.exp(-all_logits_out))  # sigmoid

# ---------------------------------------------------------------------------
# 4. Build final analysis DataFrame
# ---------------------------------------------------------------------------
inference_df = test_df.copy()

if NUM_CLASSES > 1:
    # Multi-target: one logit & prob column per target
    for i, col in enumerate(TARGET_COLUMNS):
        inference_df[f"logit_{col}"] = all_logits_out[:, i]
        inference_df[f"prob_{col}"] = all_probs[:, i]

    # Per-target AUC
    has_all_targets = all(col in test_df.columns for col in TARGET_COLUMNS)
    if has_all_targets:
        print()
        auc_results = {}
        for i, col in enumerate(TARGET_COLUMNS):
            gt = test_df[col].values.astype(np.float32)
            preds = all_probs[:, i]
            try:
                auc = roc_auc_score(gt, preds)
            except ValueError:
                auc = float("nan")
            auc_results[col] = auc
            print(f"AUC [{col}]: {auc:.6f}  |  "
                  f"pos rate (gt): {gt.mean():.4%}  |  "
                  f"pos rate (pred>0.5): {(preds >= 0.5).mean():.4%}")
        print(f"\nMean AUC: {np.nanmean(list(auc_results.values())):.6f}")
    else:
        missing = [c for c in TARGET_COLUMNS if c not in test_df.columns]
        print(f"\nMissing target columns in test.pkl: {missing} — skipping AUC.")

    display_cols = ["sess_seq_len"] + [f"logit_{c}" for c in TARGET_COLUMNS] + [f"prob_{c}" for c in TARGET_COLUMNS]

else:
    # Single target (original behavior)
    inference_df["logit"] = all_logits_out
    inference_df["prob"] = all_probs

    if TARGET_COLUMNS[0] in test_df.columns:
        gt = test_df[TARGET_COLUMNS[0]].values.astype(np.float32)
        try:
            auc = roc_auc_score(gt, all_probs)
            print(f"\nAUC: {auc:.6f}")
        except ValueError:
            print("\nAUC: could not compute (single class?)")
        print(f"Positive rate (ground truth): {gt.mean():.4%}")
        print(f"Positive rate (pred > 0.5):   {(all_probs >= 0.5).mean():.4%}")
    else:
        print(f"\nNo '{TARGET_COLUMNS[0]}' column in test.pkl — skipping AUC.")

    display_cols = ["sess_seq_len", "logit", "prob"]

print(f"\nInference DataFrame shape: {inference_df.shape}")
print(inference_df[display_cols].head(10))

inference_df.to_pickle("inference_results.pkl")
print(f"\nSaved inference results to inference_results.pkl")